# SAMI — Notebook 2 · Comportamiento general y necesidades

**Todo lo complejo que no es NLP:** cruces de variables, tendencias temporales, primeras voces cualitativas, necesidades más solicitadas, profundidad de uso. El notebook operativo — empieza a mostrar fricción.

*Las categorías de necesidad requieren clasificación (NLP) y viven en el Notebook 3, junto con el clustering, la comparación con la clasificación original y el sentimiento.*

## Setup

In [ ]:
# Imports. Collapsed on purpose -- no analysis here.
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from collections import Counter
from wordcloud import WordCloud

import geopandas as gpd
import contextily as cx
from shapely.geometry import Point
from adjustText import adjust_text

import mmc_data, mmc_entities

In [ ]:
# Temporary neutral styling -- brand palette removed; matplotlib defaults for now.
# These names shadow the old palette so moved cells run unchanged. Re-add the
# real palette later by replacing THIS cell (nothing else references the brand).
_CYCLE = plt.rcParams["axes.prop_cycle"].by_key()["color"]   # matplotlib defaults
PRIMARY = _CYCLE[0]
BLUE_SEQ = BLUES = _CYCLE
EARTH    = _CYCLE
CAT      = _CYCLE
# brand color names -> matplotlib defaults, so cells referencing them still run
AGUA, ARBOL, AMEBA, MADERA, HONGO, NEGRO = (
    _CYCLE[0], _CYCLE[2], _CYCLE[4], _CYCLE[1], "#dddddd", "black")
INK = INK2 = "black"
MUTED = "gray"
GRID  = "#cccccc"
SURFACE = "white"

def cat_colors(n):
    """n distinct matplotlib default colors (categorical)."""
    return [_CYCLE[i % len(_CYCLE)] for i in range(n)]

def bar_colors(n):
    """alias of cat_colors -- n distinct matplotlib default colors."""
    return [_CYCLE[i % len(_CYCLE)] for i in range(n)]

def seq_colors(n):
    """single default color repeated (ordered magnitude -- one hue for now)."""
    return [_CYCLE[0]] * n

def pct_count_autopct(values, min_pct=3.0):
    """Pie label 'xx.x%\n(n)'; blank under min_pct. Label formatter, not color."""
    total = float(sum(values))
    def _fmt(pct):
        if pct < min_pct:
            return ""
        return f"{pct:.1f}%\n({int(round(pct/100*total))})"
    return _fmt

### Data

Two cleaning conventions coexist here. `df` / `meal` come from the EDA cleaning (rich display columns: `city_display`, `nationality_clean`, durations, destinations). `msgs` is the message-level spine from `mmc_data` (one row per user turn).

In [ ]:
DATA_PATH = '../data_&_docs/MMC_bot_responses_Grupo_nuevo_1783087815.xlsx'

df = pd.read_excel(DATA_PATH, sheet_name='mmc bot - responses', header=2)
df = df.dropna(how='all').reset_index(drop=True)

# One row has no Name/Timestamp/other field except a stray "Questions per
# user" = 388 -- a spreadsheet artifact, not a real interaction. Drop it.
df = df[df['Name'].notna()].reset_index(drop=True)

df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Questions per user'] = pd.to_numeric(df['Questions per user'], errors='coerce')

# Consolidate _other columns: fall back to free-text when main option is NaN/"Otra"
df['city_display'] = df.apply(
    lambda r: r['City_other'] if r['City'] == 'Otra' else r['City'], axis=1
)
df['nationality_display'] = df.apply(
    lambda r: r['Nationality_other'] if pd.isna(r['Nationality']) else r['Nationality'], axis=1
)

# Canonicalize city: city_display is very messy -- the same city appears under
# many spellings ("Bogotá"/"Bogota"/"Bogotá D.C"/"Colombia Bogotá"), with mixed
# case ("Santa marta"), and with department suffixes ("Soacha Cundinamarca",
# "Barranquilla Atlántico"). We match on an accent/case-insensitive key and map
# every variant to one canonical name, and drop entries that are departments or
# a country rather than a city ("Colombia", "Cundinamarca", "Antioquia", "9").
import unicodedata

def _city_key(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return s.lower().strip().rstrip('.').strip()

CITY_CANON_KEYS = {
    'medellin': 'Medellín',
    'bogota': 'Bogotá', 'bogota dc': 'Bogotá', 'bogota d.c': 'Bogotá',
    'bogota d c': 'Bogotá', 'colombia bogota': 'Bogotá', 'bogota zipaquira': 'Bogotá',
    'cucuta': 'Cúcuta',
    'santa marta': 'Santa Marta',
    'soacha': 'Soacha', 'soacha cundinamarca': 'Soacha', 'soacha condinamarca': 'Soacha',
    'soacha, cundinamarca': 'Soacha',
    'cali': 'Cali',
    'necocli': 'Necoclí',
    'barranquilla': 'Barranquilla', 'barranquilla atlantico': 'Barranquilla',
    'riohacha': 'Riohacha', 'riohacha la guajira': 'Riohacha',
    'turbo': 'Turbo', 'turbo antioquia': 'Turbo',
    'ipiales': 'Ipiales', 'bucaramanga': 'Bucaramanga', 'maicao': 'Maicao',
    'cartagena': 'Cartagena', 'tumaco': 'Tumaco', 'pasto': 'Pasto',
    'pereira': 'Pereira', 'galapa': 'Galapa',
}
CITY_DROP = {'9', 'colombia', 'cundinamarca', 'antioquia'}

def _canon_city(c):
    if not isinstance(c, str):
        return c
    k = _city_key(c)
    if k in CITY_DROP:
        return np.nan
    return CITY_CANON_KEYS.get(k, c.strip())

df['city_display'] = df['city_display'].map(_canon_city)

# Canonicalize nationality: merge duplicate spellings of the same country so
# e.g. "Colombia" and "Colombiana" are ONE category, and drop non-country junk.
NATIONALITY_CANON = {
    'Colombiana': 'Colombia', 'Soy colombovenezolana': 'Venezuela',
    'Haiti': 'Haití', 'Panama': 'Panamá', 'Peru': 'Perú', 'Mexico': 'México',
}
INVALID_NATIONALITIES = {'1', '3', '4', 'Valyria'}
_nat = df['nationality_display'].astype('string').str.strip().replace(NATIONALITY_CANON)
df['nationality_clean'] = _nat.where(~_nat.isin(INVALID_NATIONALITIES))

# City_duration is free-text with ~20 spelling/format variants of the same 5
# duration ranges. DURATION_MAP normalizes to 5 ranges + "No especifica".
DURATION_ORDER = [
    'Menos de 1 mes', 'Entre 1 y 3 meses', 'Entre 4 y 6 meses',
    'Entre 7 meses y 1 año', 'Más de 1 año', 'No especifica',
]
DURATION_MAP = {
    'Más de 1 año': 'Más de 1 año', 'Menos de 1 mes': 'Menos de 1 mes',
    'Entre 1 y 3 meses': 'Entre 1 y 3 meses',
    'Entre 7 meses y 1 año': 'Entre 7 meses y 1 año',
    'Entre 4 y 6 meses': 'Entre 4 y 6 meses',
    '2 años': 'Más de 1 año', '15 días': 'Menos de 1 mes', '8 años': 'Más de 1 año',
    '5 años': 'Más de 1 año', '3 años': 'Más de 1 año',
    'Ya tengo una semana': 'Menos de 1 mes', '7 años': 'Más de 1 año',
    'Vendo': 'No especifica', '8ños': 'Más de 1 año', '7 años 9 meses': 'Más de 1 año',
    '5 anos': 'Más de 1 año', '2 año': 'Más de 1 año', 'Santuario': 'No especifica',
    'Acabo de llegar': 'Menos de 1 mes', '3 meses': 'Entre 1 y 3 meses',
    'Norte de Santander': 'No especifica',
}
df['city_duration_clean'] = df['City_duration'].map(DURATION_MAP)

print(f"Shape: {df.shape}")
df.dtypes

In [ ]:
# Source spreadsheets: MEAL feedback form and chatbot interaction log
MEAL_PATH = '../data_&_docs/MMC_MEAL_Group_Title_1783087939.xlsx'
RESP_PATH = '../data_&_docs/MMC_bot_responses_Grupo_nuevo_1783087815.xlsx'

# header=2: the sheet has two banner rows above the real column headers
meal = pd.read_excel(MEAL_PATH, sheet_name='mmc-meal', header=2)
meal = meal.dropna(how='all').reset_index(drop=True)  # drop blank rows left by the banner

# Replace the long Spanish question text with short, code-friendly column names
meal.columns = [
    'Name', 'Timestamp',
    'usefulness_rating', 'would_recommend',
    'recommendation_text', 'discovery_channel', 'discovery_other',
]
meal['Timestamp'] = pd.to_datetime(meal['Timestamp'], errors='coerce')

# Map the Spanish Likert labels to a 1-5 numeric scale so we can average / trend
RATING_MAP = {
    'Muy útil': 5, 'Útil': 4,
    'Medianamente útil': 3, 'Poco útil': 2, 'Nada útil': 1,
}
meal['rating_num'] = meal['usefulness_rating'].map(RATING_MAP)

print(f"MEAL records: {len(meal)}")
meal.head()

In [ ]:
# Message-level spine: one row per user turn (mmc_data cleaning: city_canon, phone).
_resp = mmc_data.load_responses()
msgs = mmc_data.load_messages(_resp)
print(f"df users: {len(df)}  |  meal: {len(meal)}  |  messages: {len(msgs)}")

## 1. Cross-cuts — demographics & geography

*Patterns that only appear when variables are crossed — invisible to the univariate EDA.*

### 1.1 How settled are users, city by city?

In [ ]:
# For each major city, what share are recent arrivals vs long-settled? A 100%
# stacked bar makes the *mix* comparable across cities regardless of sample size.
top_cities = df['city_display'].value_counts().head(8).index
cross = (
    df[df['city_display'].isin(top_cities)]
    .dropna(subset=['city_display', 'city_duration_clean'])
    .groupby(['city_display', 'city_duration_clean']).size()
    .unstack(fill_value=0)
)
cols = [c for c in DURATION_ORDER if c in cross.columns]
cross_pct = cross[cols].div(cross[cols].sum(axis=1), axis=0) * 100
cross_pct = cross_pct.reindex([c for c in top_cities if c in cross_pct.index])

# ordered light->dark ramp for the duration buckets; grey for "No especifica"
_seq = seq_colors(len([c for c in cols if c != 'No especifica']))
_i = 0
dur_colors = []
for c in cols:
    if c == 'No especifica':
        dur_colors.append(MUTED)
    else:
        dur_colors.append(_seq[_i]); _i += 1

ax = cross_pct.plot(kind='bar', stacked=True, color=dur_colors,
                    figsize=(11, 5), edgecolor=SURFACE, linewidth=0.6, width=0.8)
ax.set_title('How settled are users, city by city? (% by length of residence)')
ax.set_xlabel('')
ax.set_ylabel('Share of users (%)')
ax.set_ylim(0, 100)
ax.legend(title='Time in city', bbox_to_anchor=(1.01, 1), loc='upper left',
          fontsize=8, frameon=False)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

### 1.2 Where are the users? (map)

In [ ]:
# Step 1 -- build the city bubbles (hard-coded centroids + user counts).
# city_display is already canonicalized in Section 1, so each city is one name.
CITY_COORDS = {   # (lon, lat)
    "Medellín": (-75.5812, 6.2442), "Cúcuta": (-72.5078, 7.8939),
    "Ipiales": (-77.6453, 0.8297), "Necoclí": (-76.7858, 8.4280),
    "Bogotá": (-74.0721, 4.7110), "Soacha": (-74.2170, 4.5790),
    "Barranquilla": (-74.7964, 10.9639), "Cali": (-76.5320, 3.4516),
    "Turbo": (-76.7305, 8.0979), "Santa Marta": (-74.2139, 11.2408),
    "Tumaco": (-78.8168, 1.8070), "Maicao": (-72.2427, 11.3703),
    "Riohacha": (-72.9072, 11.5442), "Galapa": (-74.8860, 10.8990),
    "Cartagena": (-75.5144, 10.3910), "Bucaramanga": (-73.1198, 7.1293),
    "Pereira": (-75.6942, 4.8087), "Armenia": (-75.6810, 4.5339),
    "Manizales": (-75.5174, 5.0703), "Pasto": (-77.2811, 1.2136),
    "Villavicencio": (-73.6354, 4.1420),
}
city_counts = (
    df['city_display'][df['city_display'].isin(CITY_COORDS)].value_counts()
    .rename_axis('city').reset_index(name='count')
)
city_counts['geometry'] = city_counts['city'].map(lambda c: Point(CITY_COORDS[c]))
cities_gdf = gpd.GeoDataFrame(city_counts, geometry='geometry', crs='EPSG:4326').to_crs(3857)
cities_gdf[['city', 'count']].head()

In [ ]:
# Step 2 -- plot the bubbles over a real basemap so the geography is grounded.
mx = cities_gdf['count'].max()
cities_gdf['ms'] = np.sqrt(cities_gdf['count'] / mx) * 1500 + 60

fig, ax = plt.subplots(figsize=(7.5, 9))
cities_gdf.plot(ax=ax, markersize=cities_gdf['ms'], color=PRIMARY, alpha=0.85,
                edgecolor='white', linewidth=0.9, zorder=3)

# Equal padding in metres + equal aspect so the map keeps true proportions
# (otherwise the frame stretches the country sideways).
minx, miny, maxx, maxy = cities_gdf.total_bounds
pad = max(maxx - minx, maxy - miny) * 0.08
ax.set_xlim(minx - pad, maxx + pad)
ax.set_ylim(miny - pad, maxy + pad)
ax.set_aspect('equal')
cx.add_basemap(ax, source=cx.providers.CartoDB.PositronNoLabels, crs=cities_gdf.crs)

texts = [
    ax.text(r.geometry.x, r.geometry.y, f"{r['city']} ({r['count']})",
            fontsize=7.5, color=INK, zorder=5,
            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.75, ec='none'))
    for _, r in cities_gdf.iterrows()
]
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color=INK2, lw=0.6))

ax.set_axis_off()
ax.set_title('User distribution across Colombia (bubble size = users)', fontsize=12)
plt.tight_layout()
plt.show()

### 1.3 Migration routes — origin → destination

In [ ]:
# Step 1 -- build origin (previous country) and destination bubbles.
COUNTRY_CANON = {'United States': 'Estados Unidos', 'Peru': 'Perú',
                 'Panama': 'Panamá', 'Brazil': 'Brasil'}
NON_COUNTRY = {'Valyria', 'Valledupar'}
COUNTRY_COORDS = {   # (lon, lat)
    'Colombia': (-74.3, 4.6), 'Venezuela': (-66.6, 6.4), 'Chile': (-71.5, -35.7),
    'Brasil': (-51.9, -14.2), 'Cuba': (-79.5, 21.5), 'Haití': (-72.3, 18.9),
    'Argentina': (-63.6, -38.4), 'Panamá': (-80.0, 8.4), 'Ecuador': (-78.2, -1.8),
    'Perú': (-75.0, -9.2), 'Estados Unidos': (-98.5, 39.8),
    'Luxemburgo': (6.1, 49.8), 'Bulgaria': (25.5, 42.7), 'España': (-3.7, 40.4),
    'Alemania': (10.4, 51.2),
}

def canon_country(s):
    s = str(s).strip().title()
    return COUNTRY_CANON.get(s, s)

# Origins: previous country + Colombian nationals (who never crossed a border)
orig = df['Prev_country_other'].dropna().map(canon_country)
orig = orig[~orig.isin(NON_COUNTRY)].value_counts()
n_col = int((df['nationality_clean'] == 'Colombia').sum())
if n_col:
    orig['Colombia'] = orig.get('Colombia', 0) + n_col
# Destinations
dest = df['Destination_Country'].dropna().map(canon_country)
dest = dest[~dest.isin(NON_COUNTRY)].value_counts()

def build_gdf(counts):
    rows = [{'country': c, 'count': int(n), 'geometry': Point(COUNTRY_COORDS[c])}
            for c, n in counts.items() if c in COUNTRY_COORDS]
    return gpd.GeoDataFrame(rows, geometry='geometry', crs='EPSG:4326')

orig_gdf = build_gdf(orig)
dest_gdf = build_gdf(dest)
# Countries off the Americas frame (a few European origins) -- noted under the map
off_map = [(c, int(n)) for c, n in orig.items()
           if c in COUNTRY_COORDS and COUNTRY_COORDS[c][0] > -30]
orig_gdf = orig_gdf[orig_gdf.geometry.x < -30].to_crs(3857)
dest_gdf = dest_gdf.to_crs(3857)
print("Origins:", orig.to_dict())
print("Destinations:", dest.to_dict())

In [ ]:
# Step 2 -- one map, colour key = origin vs destination, size = users.
ORIG_COLOR, DEST_COLOR = '#6e824a', '#6D8BDB'   # brand green = origin, blue = destination
mx = max(orig_gdf['count'].max(), dest_gdf['count'].max())

def msize(g):
    return np.sqrt(g['count'] / mx) * 1300 + 45

xs = list(orig_gdf.geometry.x) + list(dest_gdf.geometry.x)
ys = list(orig_gdf.geometry.y) + list(dest_gdf.geometry.y)
# Where a country is both an origin AND a destination (Colombia, Venezuela,
# Chile) the two bubbles sit on the same point. Nudge origins up-left and
# destinations down-right by a small fixed vector so both stay visible.
off = min(max(xs) - min(xs), max(ys) - min(ys)) * 0.028
ox, oy = orig_gdf.geometry.x - off, orig_gdf.geometry.y + off * 0.6
dx, dy = dest_gdf.geometry.x + off, dest_gdf.geometry.y - off * 0.6

fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(dx, dy, s=msize(dest_gdf), color=DEST_COLOR, alpha=0.8,
           edgecolor='white', linewidth=0.9, zorder=3)
ax.scatter(ox, oy, s=msize(orig_gdf), color=ORIG_COLOR, alpha=0.8,
           edgecolor='white', linewidth=0.9, zorder=4)

pad = max(max(xs) - min(xs), max(ys) - min(ys)) * 0.06
ax.set_xlim(min(xs) - pad, max(xs) + pad)
ax.set_ylim(min(ys) - pad, max(ys) + pad)
ax.set_aspect('equal')
cx.add_basemap(ax, source=cx.providers.CartoDB.PositronNoLabels, crs='EPSG:3857')

texts = []
for gx, gy, g, col in [(dx, dy, dest_gdf, DEST_COLOR), (ox, oy, orig_gdf, ORIG_COLOR)]:
    for (x_, y_), (_, r) in zip(zip(gx, gy), g.iterrows()):
        texts.append(ax.text(x_, y_, f"{r['country']} ({r['count']})",
                             fontsize=7, color=col, fontweight='bold', zorder=6,
                             bbox=dict(boxstyle='round,pad=0.15', fc='white', alpha=0.75, ec='none')))
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color=INK2, lw=0.5))

handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=ORIG_COLOR,
                  markersize=11, label='Origin (previous country)'),
           Line2D([0], [0], marker='o', color='w', markerfacecolor=DEST_COLOR,
                  markersize=11, label='Destination country')]
ax.legend(handles=handles, loc='lower left', frameon=True, fontsize=9)
ax.set_axis_off()
ax.set_title('Migration routes: where users came from and where they head next\n'
             '(bubble size = users; Colombia appears as both origin and destination)',
             fontsize=12)
if off_map:
    note = 'Also reported as origin, off-map (Europe): ' + ', '.join(f"{c} ({n})" for c, n in off_map)
    ax.text(0.5, -0.02, note, transform=ax.transAxes, ha='center', va='top',
            fontsize=8, color=MUTED)
plt.tight_layout()
plt.show()

### 1.4 Gender composition by nationality

In [ ]:
# Does gender balance differ by nationality? A 100% stacked bar shows the gender
# *composition* within each nationality (Top 5 + Otros) -- readable even where
# the raw counts are tiny (a plain count heatmap here is mostly zeros).
top_nat = df['nationality_clean'].value_counts().head(5).index.tolist()
tmp = df.dropna(subset=['Gender', 'nationality_clean']).copy()
tmp['nat_grp'] = tmp['nationality_clean'].where(tmp['nationality_clean'].isin(top_nat), 'Otros')
order = top_nat + ['Otros']
comp = tmp.groupby(['nat_grp', 'Gender']).size().unstack(fill_value=0).reindex(order).fillna(0)
comp_pct = comp.div(comp.sum(axis=1), axis=0) * 100

ax = comp_pct.plot(kind='barh', stacked=True, color=cat_colors(comp_pct.shape[1]),
                   figsize=(9, 4.5), edgecolor=SURFACE, linewidth=0.8, width=0.75)
totals = comp.sum(axis=1)
for i, tot in enumerate(totals.values):
    ax.text(101, i, f"n={int(tot)}", va='center', fontsize=8, color=MUTED)
ax.set_xlim(0, 100)
ax.invert_yaxis()
ax.set_xlabel('Share of users (%)')
ax.set_ylabel('')
ax.legend(title='Gender', bbox_to_anchor=(1.06, 1), loc='upper left', fontsize=8, frameon=False)
ax.set_title('Gender composition within each nationality (Top 5 + Otros)')
plt.tight_layout()
plt.show()

### 1.5 Engagement by city

In [ ]:
# Which cities have the most engaged users (avg questions per user)? Each bar is
# annotated with n= (users behind the average) so thin samples aren't over-read.
top_cities = df['city_display'].value_counts().head(8).index
city_eng = (
    df[df['city_display'].isin(top_cities)]
    .groupby('city_display')['Questions per user']
    .agg(mean='mean', n='count').sort_values('mean', ascending=False)
)
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(city_eng.index, city_eng['mean'], color=PRIMARY)
for bar, n in zip(bars, city_eng['n']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f"{bar.get_height():.1f}\n(n={n})", ha='center', va='bottom', fontsize=8, color=INK2)
ax.set_ylim(0, city_eng['mean'].max() * 1.3)
ax.set_ylabel('Avg questions per user')
ax.grid(True, axis='y')
ax.set_xticklabels(city_eng.index, rotation=20, ha='right')
ax.set_title('Which cities have the most engaged users? (top 8)')
plt.tight_layout()
plt.show()

### 1.6 Age by intended destination

In [ ]:
# Does the age profile differ by intended destination country?
top_dests = df['Destination_Country'].value_counts().head(5).index.tolist()
subset = df[df['Destination_Country'].isin(top_dests)]
groups = [subset.loc[subset['Destination_Country'] == d, 'Age'].dropna().values for d in top_dests]
groups = [(g if len(g) else np.array([np.nan])) for g in groups]

fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(groups, patch_artist=True, vert=True,
                medianprops=dict(color=INK, linewidth=2),
                whiskerprops=dict(color=INK2), capprops=dict(color=INK2),
                flierprops=dict(marker='o', markersize=4, markerfacecolor=MUTED,
                                markeredgecolor='none', alpha=0.5))
for patch, color in zip(bp['boxes'], cat_colors(len(top_dests))):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax.set_xticklabels(top_dests, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Age (years)')
ax.grid(True, axis='y')
ax.set_title('Does age differ by intended destination? (top 5 destinations)')
plt.tight_layout()
plt.show()

## 2. Trends over time

### 2.1 Daily chatbot usage

In [ ]:
# How has daily chatbot usage evolved (growth, drop-offs, spikes)?
timeline = df.dropna(subset=['Timestamp']).set_index('Timestamp').resample('D').size()
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(timeline.index, timeline.values, color=BLUE_SEQ[0], alpha=0.5)
ax.plot(timeline.index, timeline.values, color=PRIMARY, linewidth=1.8)
ax.set_xlabel('Date')
ax.set_ylabel('Interactions')
ax.grid(True, axis='y')
ax.set_title('Daily user interactions over time')
plt.tight_layout()
plt.show()

### 2.2 MEAL responses over time

In [ ]:
# Left: cumulative reach of the survey. Right: ratings over time + rolling mean.
timeline = meal.dropna(subset=['Timestamp']).set_index('Timestamp').resample('D').size()
cumulative = timeline.cumsum()

meal_sorted = meal.dropna(subset=['Timestamp', 'rating_num']).sort_values('Timestamp')
rolling_mean = meal_sorted.set_index('Timestamp')['rating_num'].rolling('14D', min_periods=2).mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].fill_between(cumulative.index, cumulative.values, color=BLUE_SEQ[0], alpha=0.5)
axes[0].plot(cumulative.index, cumulative.values, color=PRIMARY, linewidth=1.8)
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Cumulative responses')
axes[0].grid(True, axis='y')
axes[0].set_title('Feedback collected over time (cumulative)')

axes[1].scatter(meal_sorted['Timestamp'], meal_sorted['rating_num'],
                color=PRIMARY, alpha=0.6, s=55, edgecolors='white', linewidths=0.5,
                label='Individual rating')
axes[1].plot(rolling_mean.index, rolling_mean.values, color='#6e824a', linewidth=2,
             label='14-day rolling mean')
axes[1].set_ylim(0.5, 5.5)
axes[1].set_yticks([1, 2, 3, 4, 5])
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Usefulness rating (1–5)')
axes[1].grid(True, axis='y')
axes[1].legend(fontsize=8, frameon=False)
axes[1].set_title('Is satisfaction trending up or down?')

for ax in axes:
    plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
plt.tight_layout()
plt.show()

## 3. Qualitative voices

*The first human break in the notebook — what users say in their own words.*

### 3.1 Word cloud of recommendations

In [ ]:
# Common Spanish stopwords to exclude so the cloud highlights meaningful themes.
ES_STOPWORDS = {
    'de', 'la', 'el', 'en', 'y', 'a', 'que', 'no', 'es', 'se', 'los', 'las',
    'con', 'por', 'un', 'una', 'para', 'más', 'del', 'al', 'lo', 'le', 'su',
    'me', 'mi', 'como', 'muy', 'pero', 'si', 'ya', 'este', 'esta', 'son', 'hay',
    'fue', 'ser', 'tiene', 'han', 'he', 'nos', 'sus', 'todo', 'también',
}
rec_text = meal['recommendation_text'].dropna()
all_words = ' '.join(rec_text.str.lower()).split()
words = [w.strip('.,;:!?()\"\'') for w in all_words
         if len(w) > 3 and w.strip('.,;:!?()\"\'') not in ES_STOPWORDS]
word_freq = Counter(words)

# Colour words from the categorical palette so the cloud matches the notebook.
def _brand_color_func(word, font_size, position, orientation, random_state, **kwargs):
    return CAT[random_state.randint(0, len(CAT) - 1)]

wc = WordCloud(width=900, height=450, background_color=SURFACE,
               color_func=_brand_color_func, prefer_horizontal=0.9
               ).generate_from_frequencies(word_freq)

fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Most common words in user recommendations', fontsize=12)
plt.tight_layout()
plt.show()

### 3.2 Free-text recommendations (thematic read)

**What this shows.** What respondents wrote when asked what could be improved, after filtering out non-answers (e.g. "no", "ninguna"). **Why it matters.** Free text is the only place respondents can raise something the closed-ended questions didn't anticipate.

> **Technical note — qualitative only.** With few substantive rows, the free text is read manually below rather than topic-modeled or embedded — there isn't enough volume for that to be meaningful.

In [ ]:
recs = meal["recommendation_text"].dropna().astype(str)
recs = recs[~recs.str.strip().str.lower().isin(["no", "ninguna", "ninguno", "nada"])]
print(f"{len(recs)} substantive recommendations\n")
for t in recs.head(20):
    print("•", t)

## 4. Most-requested needs (message level)

*Dictionary matching of procedures & institutions — not NLP clustering.*

### 4.1 Procedures & institutions mentioned

In [ ]:
counts = mmc_entities.entity_counts(msgs["message"])
fig, ax = plt.subplots(figsize=(8, 6))
top = counts.head(15).iloc[::-1]
ax.barh(top.index, top.values, color=bar_colors(len(top)))
ax.set_title("Most-mentioned procedures & institutions")
ax.set_xlabel("messages mentioning"); plt.tight_layout()

# per-entity boolean columns for cross-tabs
for ent in counts.index:
    msgs["ent_" + ent] = msgs["message"].fillna("").map(
        lambda t, e=ent: e in mmc_entities.extract_entities(t))

### 4.2 Messages by city

In [ ]:
top_cities = msgs.loc[msgs["city_canon"] != "Otra", "city_canon"].value_counts().head(10)
fig, ax = plt.subplots(figsize=(8, 6))
order = top_cities.iloc[::-1]
ax.barh(order.index, order.values, color=bar_colors(len(order)))
ax.set_title("Messages by city (top 10)")
ax.set_xlabel("messages"); plt.tight_layout()

## 5. Engagement depth

*How far users go — the bridge to Notebook 3, where we ask **what** they asked about and **how** they felt.*

In [ ]:
per_user = msgs.groupby("phone")["n_msgs_user"].first()
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(per_user, bins=range(1, int(per_user.max()) + 2), color=AGUA, edgecolor=NEGRO)
ax.set_title("Messages per user"); ax.set_xlabel("messages"); ax.set_ylabel("users")
ax.set_xlim(1, 20); plt.tight_layout()
print(f"median {per_user.median():.0f}  |  single-message share "
      f"{(per_user<=1).mean()*100:.1f}%  |  90th pct {per_user.quantile(.9):.0f}")